In [0]:
%python
df_orders = spark.read.csv(
    "/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

df_orders.show(5)

In [0]:
df_orders.createOrReplaceTempView("orders")

In [0]:
%sql
SELECT * 
FROM orders
LIMIT 10;

In [0]:
%sql

SELECT order_status, COUNT(order_id) as total
FROM orders
GROUP BY order_status;

In [0]:
%python

df_orders.write \
.format("delta") \
.mode("overwrite") \
.save("/Volumes/workspace/default/olist_raw/orders_delta")

In [0]:
%sql
SELECT *
FROM delta.`/Volumes/workspace/default/olist_raw/orders_delta`
LIMIT 10;

In [0]:
%sql
CREATE TABLE orders_delta
USING DELTA
AS SELECT * 
FROM orders;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.bronze.orders
USING DELTA
AS SELECT * 
FROM csv.`/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv`

In [0]:
%python
files_info = dbutils.fs.ls('/Volumes/workspace/default/olist_raw/')

for file in files_info:
    if file.name.endswith(".csv"):

        file_name = file.name.replace(".csv","")

        df = spark.read.csv(
            file.path,
            header=True,
            inferSchema=True,
            multiLine=True,
            escape='"'
        )

        df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"workspace.bronze.{file_name}")


### Finding `olist_order_reviews` parsing problem

In [0]:
df_reviews = spark.read.csv(
    "/Volumes/workspace/default/olist_raw/olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=True,
    multiLine=True
)

df_reviews.printSchema()

In [0]:
print(df_reviews.count())

In [0]:
display(
    spark.table("workspace.bronze.olist_order_reviews_dataset")
    .select(
        "review_id",
        "order_id",
        "review_score",
        "review_creation_date",
        "review_answer_timestamp"
    )
    .limit(20)
)

In [0]:
df_bronze = spark.table(
    "workspace.bronze.olist_order_reviews_dataset"
)

print("Bronze:", df_bronze.count())
print("Direct read:", df_reviews.count())

In [0]:
df_bronze.printSchema()

In [0]:
display(
    df_bronze
    .filter(df_bronze.review_score.cast("int").isNull())
    .select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date"
    )
    .limit(20)
)

In [0]:
df_reviews_test_2 = spark.read.csv(
    "/Volumes/workspace/default/olist_raw/olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

In [0]:
df_reviews_test_2.printSchema()
print(df_reviews_test_2.count())

In [0]:
display(
    df_reviews_test_2
    .filter(df_reviews_test_2.review_score.cast("int").isNull())
    .select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date"
    )
    .limit(20)
)

CSV Ingestion — multiLine and escape

During the Olist ingestion process, we identified CSV parsing issues caused by text fields containing line breaks and quotation marks.

`multiLine=True` -> Allows the CSV parser to handle line breaks inside quoted fields as part of the same record instead of interpreting them as a new row.

`escape='"'` -> Defines the double quote (") as the escape character, allowing quotation marks inside text fields to be interpreted correctly.

The ingestion configuration was therefore adjusted to:

df = spark.read.csv(
    file.path,
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

Without these options, some review records were incorrectly parsed, causing values from different columns to shift into the wrong columns. For example, text from review_comment_message could end up in review_id or review_score.

Key lesson: CSV ingestion requires understanding the source file's formatting rules; read.csv() defaults may not be sufficient for real-world datasets.